# ImplementForge — FreshCart Under Pressure
## COSE 278: Implementing Systems — Day 4

**Run cells in order. Do not skip cells.**

Your instructor's game engine loads automatically from GitHub.
Do not modify cells marked "do not modify".


## ⚙️ CELL 0 — Setup
*Run this first. Do not modify.*


In [ ]:
# Do not modify
GITHUB_BASE = "https://raw.githubusercontent.com/rickghome/278game/main"
import urllib.request, os, pickle

for filename in ["if_engine.py", "if_cards.py"]:
    urllib.request.urlretrieve(f"{GITHUB_BASE}/{filename}", filename)
    print(f"  ✅  {filename} loaded")

# normalise filenames

exec(open("if_engine.py").read())
exec(open("if_cards.py").read())
print("\n✅  ImplementForge loaded. Scroll to CELL 0b.")


## 💾 CELL 0b — Save / Restore
*Run this second. Do not modify.*


In [ ]:
# Do not modify
def save_game(game):
    with open('game_state.pkl','wb') as f: pickle.dump(game,f)
    print(f"💾  Saved — {game['team_name']}  |  Income: ${sum(game['income'].values()):,}  |  Trust: {game['trust_score']}/100")

def restore_game():
    if not os.path.exists('game_state.pkl'):
        print("❌  No saved game. Start from CELL 1.")
        return None
    with open('game_state.pkl','rb') as f: game = pickle.load(f)
    print(f"✅  Restored — {game['team_name']}  |  Income: ${sum(game['income'].values()):,}  |  Trust: {game['trust_score']}/100")
    print(f"    Seeds: {game['seeds']}")
    return game

def _run_phase(phase_cards, phase_key, decisions_list, rationales_list, game):
    """Process all cards for a phase. decisions_list and rationales_list are parallel lists."""
    import random
    env = game["environment"]
    idx = ENV_IDX[env]
    phase_income = _baseline(env)

    if not phase_cards:
        print("✅  No incidents this phase. Your earlier decisions held.")
        update_income(game, phase_key, phase_income)
        print_income_summary(game)
        save_game(game)
        return

    # Validate input counts
    active = [c for c in phase_cards if c not in ["L1_positive","L3_positive"]]
    positive = [c for c in phase_cards if c in ["L1_positive","L3_positive"]]

    # Handle positive branches first (no decision needed)
    for cid in positive:
        card = get_card(cid, game)
        protected_loss = card["income_loss"][idx]
        phase_income -= protected_loss
        game["cards_fired"].append(cid)
        print_positive(card["name"], card["message"], protected_loss,
                       "unchanged — your investment paid off")

    if not active:
        update_income(game, phase_key, phase_income)
        print_income_summary(game)
        save_game(game)
        return

    # Validate decisions
    if len(decisions_list) < len(active):
        print(f"❌  {len(active)} card(s) fired but only {len(decisions_list)} decision(s) entered.")
        print(f"    decisions = {['a'] * len(active)}  ← add one entry per card")
        return
    if len(rationales_list) < len(active):
        print(f"❌  {len(active)} rationale(s) required.")
        return
    for i,r in enumerate(rationales_list[:len(active)]):
        if not r or r.startswith("Enter"):
            print(f"❌  rationales[{i}] is required.")
            return
    for i,d in enumerate(decisions_list[:len(active)]):
        if d not in ["a","b","c"]:
            print(f"❌  decisions[{i}] must be a, b, or c.")
            return

    # Process each active card
    for i, cid in enumerate(active):
        card = get_card(cid, game)
        decision = decisions_list[i]
        rationale = rationales_list[i]
        game["cards_fired"].append(cid)
        game["decisions"][cid] = decision
        game["rationales"][cid] = rationale

        # Special case: L1 hotfix random outcome
        if cid == "L1" and decision == "b":
            ok = random.random() > 0.5
            key = "b_success" if ok else "b_fail"
            print("🎲  Hotfix:", "WORKED" if ok else "FAILED — made it worse")
            loss_triplet = card["income_loss"].get(key,(0,0,0))
            trust_delta  = card["trust_delta"].get(key,0)
        else:
            loss_triplet = card["income_loss"].get(decision,(0,0,0))
            trust_delta  = card["trust_delta"].get(decision,0)

        income_loss = loss_triplet[idx]

        # L3 locked 4-day loss
        if cid == "L3":
            income_loss += card.get("income_loss_base",{}).get("locked",(0,0,0))[idx]

        # Environment modifier
        mod = card.get("environment_modifier",{}).get(env,1.0)
        income_loss = int(income_loss * mod)

        # Seed compounding escalation (Phase 3+)
        if phase_key == "phase3":
            seed_count = count_seeds(game)
            if seed_count >= 2 and i == 0:
                income_loss = int(income_loss * 1.35)

        # Trust recovery modifier
        recovery = card.get("trust_recovery_modifier",{}).get(decision,0)
        trust_delta += recovery

        # Plant seeds
        seed = card.get("seeds",{}).get(decision)
        if seed: plant_seed(game, seed)

        phase_income -= income_loss
        update_trust(game, trust_delta)

        add_trace(game, cid, f"{phase_key}: {cid}",
                  seed_trigger=", ".join(game["seeds"]) if game["seeds"] else None,
                  severity="minor" if income_loss < 100_000 else "major",
                  next_risk=seed)
        game["facilitator_trace"][-1]["phase"] = phase_key

        print_consequence(card["name"], decision, income_loss, trust_delta,
                          consequence_note=card.get("consequence_notes",{}).get(decision,""),
                          seed_planted=seed)

    update_income(game, phase_key, max(0, phase_income))
    print_income_summary(game)
    save_game(game)

print("💾  Save/restore + phase engine ready.")
print("    Run save_game(game) after each phase to protect your progress.")


## 🏷️ CELL 1 — Team Setup


In [ ]:
TEAM_NAME = "Team Name Here"   # ← change this

game = new_game_state(TEAM_NAME)
print(f"✅  Team: {TEAM_NAME}  |  Trust: {game['trust_score']}/100")
print(f"    Scroll to CELL 2 — Frame 1.")


## 🏗️ FRAME 1 — Architecture Config

You designed FreshCart in COSE 260. Now you're building it.

**Choose one value per field. Run the cell when done.**


In [ ]:
system_profile = {

    "environment": "consumer",
    # consumer    — fast market, high volume, volatile trust
    # enterprise  — contract-driven, stable, reputational risk
    # government  — fixed budget, procurement rules, political exposure

    "team_structure": "stream_aligned",
    # stream_aligned  — small teams, end-to-end ownership, low coordination cost
    # platform        — shared services team supporting others
    # siloed          — functional departments, high handoff cost

    "build_buy_configure": "build",
    # build       — we write it ourselves
    # buy         — we purchase a product
    # configure   — we implement a vendor platform

    "primary_risk": "delivery_speed",
    # delivery_speed   — we might be too slow
    # technical_debt   — we might cut corners
    # integration      — our components might not fit together
    # vendor_lock      — we might become too dependent on third parties
    # team_capability  — we might not have the right skills

    "data_architecture": "shared_db",
    # dedicated_dbs  — each service owns its data — higher cost, clean boundaries
    # shared_db      — single shared database — lower cost, faster to build

    "coupling": "medium",   # fixed — do not change
}

is_valid, errors, warnings = validate_frame1(system_profile)
if errors:
    print("❌  Fix these errors:")
    for e in errors: print(f"    {e}")
else:
    game["frame1"] = system_profile
    game["environment"] = system_profile["environment"]
    print("✅  Frame 1 accepted.")
    for w in warnings: print(f"  {w}")
    print(f"\n    Environment:  {system_profile['environment']}")
    print(f"    Team:         {system_profile['team_structure']}")
    print(f"    Strategy:     {system_profile['build_buy_configure']}")
    print(f"    Risk:         {system_profile['primary_risk']}")
    print(f"    Data:         {system_profile['data_architecture']}")
    print(f"\n    Phase 1 baseline: ${_baseline(system_profile['environment']):,}")
    save_game(game)


## ⚠️ PHASE 1 — Build

FreshCart is under construction. Your architecture choices are meeting reality.

**CELL 3** shows your cards. **CELL 4** is where you enter your decisions.


In [ ]:
# CELL 3 — Phase 1 cards (show only)
_phase1_cards = select_build_cards(system_profile, game)
_phase1_active = [c for c in _phase1_cards if c not in ["L1_positive","L3_positive"]]

if not _phase1_cards:
    print("✅  No incidents this phase.")
else:
    print(f"{'━'*50}")
    print(f"PHASE 1 — {len(_phase1_active)} CARD(S) FIRED")
    print(f"{'━'*50}\n")
    for i, cid in enumerate(_phase1_active):
        card = get_card(cid, game)
        print(f"━━ CARD {i+1} of {len(_phase1_active)}: {card['name']} ━━\n")
        print_card(card["name"], card["scenario"], card["options"], card["flavor"])
        print()
    print(f"\n    Enter your decisions in CELL 4 below.")
    print(f"    One decision per card, in order.")


In [ ]:
# CELL 4 — Phase 1 decisions
# One entry per card that fired, in order.
# If two cards fired: decisions = ["a", "b"]

decisions   = ["a"]               # ← one entry per card
rationales  = ["Enter one sentence explaining why you chose this option"]

_run_phase(_phase1_cards, "phase1", decisions, rationales, game)
print("\n    Scroll to CELL 5 when instructed — Frame 2.")


## 🔧 FRAME 2 — Pipeline Config

You have seen Phase 1. Now configure your pipeline.

**Engineering Capacity hard cap: 100 units.**


In [ ]:
# CELL 5 — Frame 2 config

# ┌─────────────────────────────────────────┬──────────┬──────────┐
# │  Choice                                 │  Cap cost│  Speed   │
# ├─────────────────────────────────────────┼──────────┼──────────┤
# │  testing: minimal                       │     5    │  fast    │
# │  testing: standard                      │    15    │  moderate│
# │  testing: thorough                      │    30    │  slow    │
# │  deploy:  big_bang                      │     5    │  fastest │
# │  deploy:  rolling                       │    15    │  moderate│
# │  deploy:  blue_green                    │    25    │  moderate│
# │  deploy:  canary                        │    30    │  slowest │
# │  rollback: none                         │     0    │  —       │
# │  rollback: partial                      │    10    │  —       │
# │  rollback: full                         │    20    │  —       │
# │  observability: none                    │     0    │  —       │
# │  observability: basic                   │    10    │  —       │
# │  observability: full                    │    25    │  —       │
# │  change_owner: single_person            │     0    │  —       │
# │  change_owner: shared_pair              │    10    │  —       │
# │  change_owner: team_owned               │    20    │  —       │
# └─────────────────────────────────────────┴──────────┴──────────┘

pipeline_profile = {
    "testing_coverage":    "standard",    # minimal | standard | thorough
    "deployment_method":   "rolling",     # big_bang | rolling | blue_green | canary
    "rollback_plan":       "partial",     # none | partial | full
    "observability_level": "basic",       # none | basic | full
    "change_owner":        "shared_pair", # single_person | shared_pair | team_owned
}

is_valid, errors, warnings, capacity_used = validate_frame2(pipeline_profile)
print(f"Engineering Capacity: {capacity_used}/100 units\n")
if errors:
    print("❌  Fix these errors:")
    for e in errors: print(f"    {e}")
else:
    game["frame2"] = pipeline_profile
    print("✅  Frame 2 accepted.")
    for w in warnings: print(f"  {w}")
    print(f"\n    Testing:       {pipeline_profile['testing_coverage']}")
    print(f"    Deployment:    {pipeline_profile['deployment_method']}")
    print(f"    Rollback:      {pipeline_profile['rollback_plan']}")
    print(f"    Observability: {pipeline_profile['observability_level']}")
    print(f"    Change owner:  {pipeline_profile['change_owner']}")
    save_game(game)
    print("\n    Scroll to CELL 6 — Phase 2.")


## ⚠️ PHASE 2 — Live

FreshCart is live. Real load. Real users. Real consequences.

⏱️ **One card in this phase runs under a 60-second timer — decide fast.**


In [ ]:
# CELL 6 — Phase 2 cards (show only)
_phase2_cards = select_live_cards(system_profile, pipeline_profile, game)
_phase2_active = [c for c in _phase2_cards if c not in ["L1_positive","L3_positive"]]
_phase2_positive = [c for c in _phase2_cards if c in ["L1_positive","L3_positive"]]

if _phase2_positive:
    for cid in _phase2_positive:
        card = get_card(cid, game)
        print(f"\n✅ RESILIENCE PAYOFF — {card['name']}")
        print(card["message"])

if not _phase2_active:
    if not _phase2_positive:
        print("✅  No incidents this phase.")
else:
    print(f"\n{'━'*50}")
    print(f"PHASE 2 — {len(_phase2_active)} CARD(S) FIRED")
    print(f"{'━'*50}\n")
    for i, cid in enumerate(_phase2_active):
        card = get_card(cid, game)
        print(f"━━ CARD {i+1} of {len(_phase2_active)}: {card['name']} ━━\n")
        print_card(card["name"], card["scenario"], card["options"], card["flavor"])
        if card.get("timed"):
            print(f"\n    ⏱  TIMED CARD — {card['timer_seconds']} seconds. No entry = option (b).")
        print()
    print(f"\n    Enter your decisions in CELL 7.")


In [ ]:
# CELL 7 — Phase 2 decisions
# One entry per active card that fired, in order.

decisions   = ["a"]
rationales  = ["Enter one sentence explaining why you chose this option"]

_run_phase(_phase2_cards, "phase2", decisions, rationales, game)
print("\n    Scroll to CELL 8 when instructed — Frame 3.")


## 🛠️ FRAME 3 — Operations Config

You have seen Phases 1 and 2. Configure your operational layer.


In [ ]:
# CELL 8 — Frame 3 config

operations_profile = {
    "vendor_dependency":  "medium",         # low | medium | high
    "fallback_strategy":  "manual",         # none | manual | automated
    "on_call_coverage":   "business_hours", # none | business_hours | follow_the_sun | full_24x7
    "incident_response":  "runbook",        # ad_hoc | runbook | practiced
}

is_valid, errors, warnings = validate_frame3(operations_profile)
if errors:
    print("❌  Fix these errors:")
    for e in errors: print(f"    {e}")
else:
    game["frame3"] = operations_profile
    print("✅  Frame 3 accepted.")
    for w in warnings: print(f"  {w}")
    print(f"\n    Vendor:    {operations_profile['vendor_dependency']}")
    print(f"    Fallback:  {operations_profile['fallback_strategy']}")
    print(f"    On-call:   {operations_profile['on_call_coverage']}")
    print(f"    Incidents: {operations_profile['incident_response']}")
    save_game(game)
    print("\n    Scroll to CELL 9 — Phase 3.")


## ⚠️ PHASE 3 — v2 Release

**Universal requirement — same for every team:**

> FreshCart is adding a real-time delivery tracking feature.
> New Tracking Service, updated mobile API contract, change to Notification Service.
> All teams implement this. What happens next depends on what you built.


In [ ]:
# CELL 9 — Phase 3 cards (show only)
_phase3_cards = select_v2_cards(system_profile, pipeline_profile, game)
_phase3_active = [c for c in _phase3_cards if c not in ["L1_positive","L3_positive"]]
_seed_count = count_seeds(game)

if _seed_count >= 2:
    print(f"⚠  Entering Phase 3 with {_seed_count} active seeds.")
    print(f"   Severity of first card escalated.\n")

if not _phase3_active:
    print("✅  No major incidents this phase. Your earlier decisions held.")
else:
    print(f"{'━'*50}")
    print(f"PHASE 3 — {len(_phase3_active)} CARD(S) FIRED")
    print(f"{'━'*50}\n")
    for i, cid in enumerate(_phase3_active):
        card = get_card(cid, game)
        print(f"━━ CARD {i+1} of {len(_phase3_active)}: {card['name']} ━━\n")
        print_card(card["name"], card["scenario"], card["options"], card["flavor"])
        print()
    print(f"\n    Enter your decisions in CELL 10.")


In [ ]:
# CELL 10 — Phase 3 decisions

decisions   = ["a"]
rationales  = ["Enter one sentence explaining why you chose this option"]

_run_phase(_phase3_cards, "phase3", decisions, rationales, game)
print("\n    Scroll to CELL 11 when instructed — Phase 4.")


## 🚨 PHASE 4 — Scale Event

**Wait for your instructor to read the scenario aloud.**

No new config. Your architecture holds — or it doesn't.


In [ ]:
# CELL 11 — S1: The Moment of Truth
# Run after instructor reads scenario aloud.

def _calculate_s1_outcome(game):
    env   = game["environment"]
    idx   = ENV_IDX[env]
    base  = PHASE_BASELINE[idx]
    seeds = game.get("seeds",[])
    f2    = game.get("frame2",{})
    f3    = game.get("frame3",{})
    score = 0
    deploy = f2.get("deployment_method","big_bang")
    if deploy in ["canary","blue_green"]: score += 2
    elif deploy == "rolling":             score += 1
    rollback = f2.get("rollback_plan","none")
    if rollback == "full":    score += 2
    elif rollback == "partial": score += 1
    obs = f2.get("observability_level","none")
    if obs == "full":    score += 2
    elif obs == "basic": score += 1
    fallback = f3.get("fallback_strategy","none") if f3 else "none"
    if fallback == "automated": score += 2
    elif fallback == "manual":  score += 1
    ir = f3.get("incident_response","ad_hoc") if f3 else "ad_hoc"
    if ir == "practiced": score += 2
    elif ir == "runbook":  score += 1
    effective = score - len(seeds) * 0.5
    is_gov = env == "government"
    if effective >= 7:
        tier, income, td = "thriving", base + int(base*0.68), 10
        narrative = (f"Traffic spiked 35x. Systems flagged it in 90 seconds.\n"
                     f"Automated fallback activated. FreshCart degraded gracefully.\n\n"
                     f"Revenue captured: +${int(base*0.68):,} above baseline. Trust: +10.\n\n"
                     f"This is what you paid for.")
    elif effective >= 4:
        tier, income, td = "surviving", int(base*0.95), 0
        narrative = "Traffic spiked 35x. FreshCart bent but didn't break. Most revenue captured."
    elif effective >= 1:
        tier, income, td = "struggling", int(base*0.6), -15
        narrative = "Traffic spiked 35x. FreshCart struggled. Major revenue loss. Trust eroded."
    else:
        tier, income, td = "collapsing", int(base*0.2), -30
        narrative = ("Traffic spiked 35x. FreshCart collapsed under load.\n"
                     "The accumulated decisions of the past three phases produced this outcome.")
    if is_gov: income = max(income, int(base*0.4))
    seed_notes, extra_loss, extra_trust = [], 0, 0
    seed_map = {
        "P1_silent_fraud":   ((400_000,300_000,240_000), -40, "P1 silent fraud surfaces publicly."),
        "V2_mock_data":      ((200_000,150_000,120_000), -35, "V2 mock data discovered."),
        "D1_db_scaling":     ((150_000,112_000, 90_000), -10, "Shared DB saturation triggered."),
        "P3_workaround_live":((120_000, 90_000, 72_000), -10, "P3 config cascade triggered."),
        "V1_queue_risk":     ((180_000,135_000,108_000), -25, "V1 notification queue overflow."),
        "P2_shared_ownership":((100_000,75_000, 60_000), -15, "P2 repeat ownership failure."),
    }
    for seed_id, (losses, trust_hit, note) in seed_map.items():
        if seed_id in seeds:
            extra_loss += losses[idx]; extra_trust += trust_hit; seed_notes.append(note)
    income = max(0, income - extra_loss)
    td += extra_trust
    if is_gov: income = max(income, int(base*0.4))
    return tier, income, td, narrative, seed_notes

_tier, _s1_income, _s1_trust, _s1_narrative, _seed_resolutions = _calculate_s1_outcome(game)
game["cards_fired"].append("S1")
print(f"\n{'━'*50}\nPHASE 4 — THE MOMENT OF TRUTH\n{'━'*50}")
print(_s1_narrative)
if _seed_resolutions:
    print("\nSeeds resolving now:")
    for s in _seed_resolutions: print(f"  💥 {s}")
update_income(game, "phase4", _s1_income)
update_trust(game, _s1_trust)
print(f"\nOutcome tier: {_tier.upper()}")
print_income_summary(game)
save_game(game)
print("\n    Scroll to CELL 12.")


In [ ]:
# CELL 12 — S2: Trust revealed first — 30 second pause before revenue
print(f"\n{'━'*50}\nTHE BILL ARRIVES\n{'━'*50}")
print(f"\nYour Trust Score: {game['trust_score']}/100")
env_label = {"consumer":"Customer Trust","enterprise":"Stakeholder Confidence",
             "government":"Political Capital"}.get(game["environment"],"Trust")
print(f"({env_label})")
print("\n... room discussion — wait 30 seconds ...\n")
print("Run CELL 13 to reveal revenue impact and enter your decision.")


In [ ]:
# CELL 13 — S2 card display
_card_s2 = get_card("S2", game)
game["cards_fired"].append("S2")
print_card(_card_s2["name"], _card_s2["scenario"], _card_s2["options"], _card_s2["flavor"])
print("\n    Scroll to CELL 14.")


In [ ]:
# CELL 14 — S2 decision
decision_rationale_s2 = "Enter one sentence explaining why you chose this option"
decision_s2 = "a"   # ← change to a, b, or c

if decision_rationale_s2.startswith("Enter"):
    print("❌  decision_rationale_s2 required.")
elif decision_s2 not in ["a","b","c"]:
    print("❌  decision must be a, b, or c.")
else:
    game["decisions"]["S2"] = decision_s2
    game["rationales"]["S2"] = decision_rationale_s2
    env = game["environment"]
    idx = ENV_IDX[env]
    loss = _card_s2["income_loss"].get(decision_s2,(0,0,0))[idx]
    td   = _card_s2["trust_delta"].get(decision_s2,0)
    rec  = _card_s2.get("trust_recovery_modifier",{}).get(decision_s2,0)
    seed = _card_s2.get("seeds",{}).get(decision_s2)
    if seed: plant_seed(game, seed)
    game["income"]["phase4"] = max(0, game["income"]["phase4"] - loss)
    update_trust(game, td)
    if rec: print(f"    Trust recovery modifier: +{rec} applied.")
    print_consequence(_card_s2["name"], decision_s2, loss, td,
                      consequence_note=_card_s2.get("consequence_notes",{}).get(decision_s2,""),
                      seed_planted=seed)
    save_game(game)
    print("\n    Scroll to CELL 15 — government teams also check CELL 14b.")


### Government teams only — CELL 14b
*Skip if environment is not government.*


In [ ]:
# CELL 14b — S3: Political Exposure (government environment only)
# Skip this cell if your environment is not government.

if game.get("frame1",{}).get("environment") != "government":
    print("✅  Not government environment. Skip to CELL 15.")
elif not _should_fire_S3(game.get("frame1",{}), game):
    print("✅  No political exposure this phase.")
else:
    _card_s3 = get_card("S3", game)
    game["cards_fired"].append("S3")
    print_card(_card_s3["name"], _card_s3["scenario"], _card_s3["options"], _card_s3["flavor"])
    print("\n    Enter S3 decision below and re-run this cell.")
    print("    decision_s3 = 'a'  # ← set this first, then re-run")


In [ ]:
# CELL 14c — S3 decision entry (government only)
decision_rationale_s3 = "Enter one sentence explaining why you chose this option"
decision_s3 = "a"   # ← change to a, b, or c

if game.get("frame1",{}).get("environment") != "government":
    print("✅  Skip — not government.")
elif not _should_fire_S3(game.get("frame1",{}), game):
    print("✅  S3 did not fire.")
elif decision_rationale_s3.startswith("Enter"):
    print("❌  decision_rationale_s3 required.")
elif decision_s3 not in ["a","b","c"]:
    print("❌  decision must be a, b, or c.")
else:
    _card_s3 = get_card("S3", game)
    game["decisions"]["S3"] = decision_s3
    game["rationales"]["S3"] = decision_rationale_s3
    env = game["environment"]
    idx = ENV_IDX[env]
    loss = _card_s3["income_loss"].get(decision_s3,(0,0,0))[idx]
    td   = _card_s3["trust_delta"].get(decision_s3,0)
    rec  = _card_s3.get("trust_recovery_modifier",{}).get(decision_s3,0)
    game["income"]["phase4"] = max(0, game["income"]["phase4"] - loss)
    update_trust(game, td)
    if rec: print(f"    Trust recovery modifier: +{rec} applied.")
    print_consequence(_card_s3["name"], decision_s3, loss, td,
                      consequence_note=_card_s3.get("consequence_notes",{}).get(decision_s3,""))
    save_game(game)
    print("\n    Scroll to CELL 15.")


## 📊 CELL 15 — Final Summary
*Copy the output and give it to your instructor.*


In [ ]:
print_final_summary(game)


## 🔁 CELL 16 — Reflection

**Fill this out before starting the second iteration.**


In [ ]:
strategy_change = {
    "what_failed":    "Describe the main thing that went wrong",
    "what_we_change": "Describe the specific config change you are making",
    "why":            "One sentence: why do you believe this change fixes the problem",
}

print("Reflection recorded:")
for k,v in strategy_change.items(): print(f"  {k}: {v}")
print("\nRun CELL 17 to begin second iteration.")


## 🔄 CELL 17 — Second Iteration Setup


In [ ]:
if any(v.startswith("Describe") for v in strategy_change.values()):
    print("❌  Complete the reflection in CELL 16 first.")
else:
    game["income"]            = {"phase1":0,"phase2":0,"phase3":0,"phase4":0}
    game["trust_score"]       = 100
    game["cards_fired"]       = []
    game["decisions"]         = {}
    game["rationales"]        = {}
    game["facilitator_trace"] = []
    game["seeds"]             = []
    game["phase"]             = 0
    print("✅  Second iteration ready.")
    print("    Scroll back to CELL 2, update your config, run all cells again.")
    print("\n    At the end the engine will ask: did your changes fix the problem?")
